![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

ADD VOLUME SPIKES AND DIFFERNCE WITH SPOT and spike voulmes for forex and indices

### Get Data For BTC Futures

https://www.quantconnect.com/datasets/binance-cryptofuture-price-data

In [1]:
from QuantConnect import *
from QuantConnect.Research import QuantBook
from datetime import datetime
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

qb = QuantBook()

start = pd.Timestamp(2022, 5, 1, tz="UTC") 
end = pd.Timestamp(2023, 5, 1, tz="UTC")  


print("Fetching BTCUSDT...")
future = qb.add_crypto_future('BTCUSDT', Resolution.MINUTE, Market.BINANCE)

# Get minute data
history = qb.history(future.symbol, start, end, Resolution.MINUTE, fill_forward=True)

if history.empty:
    print("No data")
else:
    # Reset and resample to 5-min
    history = history.reset_index()
    history['time_5min'] = history['time'].dt.floor('5min')
    
    btc_5min = history.groupby('time_5min').agg({
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum'
    })
    
    print(f"✓ {len(btc_5min)} bars ({btc_5min.index.min()} to {btc_5min.index.max()})")

In [2]:
import pandas as pd
import numpy as np

def create_features_btc_futures(df):
    """
    Create normalized, predictive BTC futures features for 5-min ahead prediction.
    All features are stationary/normalized and prefixed with btc_futures_.
    """
    
    features = df.copy()
    
    # === RETURNS (normalized by definition) ===
    features['ret_5min'] = features['close'].pct_change()
    features['ret_15min'] = features['close'].pct_change(3)
    features['ret_1h'] = features['close'].pct_change(12)
    features['ret_4h'] = features['close'].pct_change(48)
    
    # === VOLATILITY (rolling std of returns - normalized) ===
    features['vol_5min'] = features['ret_5min'].rolling(12).std()
    features['vol_1h'] = features['ret_5min'].rolling(48).std()
    features['vol_4h'] = features['ret_5min'].rolling(192).std()
    
    # === MOMENTUM Z-SCORES (normalized) ===
    features['ret_zscore_1h'] = (features['ret_5min'] - features['ret_5min'].rolling(12).mean()) / features['vol_5min']
    features['ret_zscore_4h'] = (features['ret_5min'] - features['ret_5min'].rolling(48).mean()) / features['vol_1h']
    
    # === VOLUME FEATURES (normalized) ===
    features['volume_ma'] = features['volume'].rolling(12).mean()
    features['volume_ratio'] = features['volume'] / (features['volume_ma'] + 1)
    features['volume_zscore'] = (features['volume'] - features['volume'].rolling(48).mean()) / features['volume'].rolling(48).std()
    
    # === PRICE STRUCTURE (normalized ratios) ===
    features['high_low_ratio'] = (features['high'] - features['low']) / features['close']
    features['close_open_ratio'] = (features['close'] - features['open']) / features['close']
    
    # === TREND INDICATORS (normalized) ===
    features['ma_12'] = features['close'].rolling(12).mean()
    features['ma_48'] = features['close'].rolling(48).mean()
    features['price_to_ma12'] = (features['close'] - features['ma_12']) / features['close']  # % distance
    features['price_to_ma48'] = (features['close'] - features['ma_48']) / features['close']
    features['ma_spread'] = (features['ma_12'] - features['ma_48']) / features['close']  # MA crossover
    
    # === RSI (bounded 0-100, normalized) ===
    delta = features['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / (loss + 1e-10)
    features['rsi'] = 100 - (100 / (1 + rs))
    features['rsi_norm'] = (features['rsi'] - 50) / 50  # Normalize to [-1, 1]
    
    # === MICROSTRUCTURE (tick-level proxies) ===
    features['hl_volatility'] = np.log(features['high'] / features['low'])  # Parkinson volatility
    features['close_position'] = (features['close'] - features['low']) / (features['high'] - features['low'] + 1e-10)  # Where close is in range
    
    # === MOMENTUM CROSS-TIMEFRAME ===
    features['mom_short_long'] = features['ret_15min'] - features['ret_1h']  # Divergence
    
    # Drop intermediate columns
    features = features.drop(['ma_12', 'ma_48', 'volume_ma', 'rsi', 'open', 'high', 'low', 'close', 'volume'], 
                             axis=1, errors='ignore')
    
    # Prefix all columns with btc_futures_
    rename_dict = {col: f'btc_futures_{col}' for col in features.columns}
    features = features.rename(columns=rename_dict)
       
    return features


# Apply to BTC data
print("="*60)
print("CREATING BTC FUTURES FEATURES")
print("="*60)

btc_features = create_features_btc_futures(btc_5min)

print(f"\n✓ BTC Features: {btc_features.shape}")
print(f"\nFeatures created:")
feature_cols = [c for c in btc_features.columns]
print(f"  {len(feature_cols)} features")
print(f"\nFirst 5 feature names:")
for col in feature_cols[:5]:
    print(f"  - {col}")

print(f"\nSample data:")
print(btc_features[['btc_futures_ret_5min', 'btc_futures_vol_5min', 'btc_futures_volume_ratio', 
                    'btc_futures_price_to_ma12', 'btc_futures_rsi_norm']].head(3))

In [3]:
import pandas as pd
import numpy as np

print("="*60)
print("BTC FEATURES - DATA QUALITY ASSESSMENT")
print("="*60)

# === STEP 1: INITIAL ASSESSMENT ===
print(f"\n1. INITIAL STATE")
print(f"   Shape: {btc_features.shape}")
print(f"   Date range: {btc_features.index.min()} to {btc_features.index.max()}")
print(f"   Duration: {(btc_features.index.max() - btc_features.index.min()).days} days")

# === STEP 2: CHECK FOR BROKEN FEATURES (>90% NaN) ===
print(f"\n2. BROKEN FEATURES CHECK (>90% NaN)")
nan_pct = (btc_features.isnull().sum() / len(btc_features)) * 100
broken = nan_pct[nan_pct > 90]

if len(broken) > 0:
    print(f"   ⚠️  Found {len(broken)} broken features:")
    for feat, pct in broken.items():
        print(f"      {feat}: {pct:.1f}% NaN")
else:
    print(f"   ✓ No broken features")

# === STEP 3: NaN DISTRIBUTION ===
print(f"\n3. NaN DISTRIBUTION")
nan_counts = btc_features.isnull().sum().sort_values(ascending=False)
has_nans = nan_counts[nan_counts > 0]

if len(has_nans) > 0:
    print(f"   Features with NaNs: {len(has_nans)}")
    print(f"\n   Top 10:")
    for feat, count in has_nans.head(10).items():
        pct = (count / len(btc_features)) * 100
        print(f"      {feat}: {count} ({pct:.1f}%)")
else:
    print(f"   ✓ No NaNs")

# === STEP 4: DETERMINE WARMUP PERIOD ===
print(f"\n4. WARMUP PERIOD ANALYSIS")

# Find first completely non-NaN row
nan_by_row = btc_features.isnull().sum(axis=1)
first_complete_row_idx = (nan_by_row == 0).idxmax() if (nan_by_row == 0).any() else None

if first_complete_row_idx:
    warmup_rows = btc_features.index.get_loc(first_complete_row_idx)
    print(f"   First complete row: index {warmup_rows} ({first_complete_row_idx})")
    print(f"   Recommended warmup: {warmup_rows} rows")
    
    # Show which features need most warmup
    print(f"\n   Warmup requirements by feature:")
    feature_warmup = {}
    for col in btc_features.columns:
        first_valid = btc_features[col].first_valid_index()
        if first_valid:
            warmup_needed = btc_features.index.get_loc(first_valid)
            feature_warmup[col] = warmup_needed
    
    # Group by warmup requirement
    warmup_groups = {}
    for feat, rows in feature_warmup.items():
        if rows not in warmup_groups:
            warmup_groups[rows] = []
        warmup_groups[rows].append(feat)
    
    for rows in sorted(warmup_groups.keys(), reverse=True)[:5]:
        print(f"      {rows} rows: {len(warmup_groups[rows])} features")
else:
    print(f"   ⚠️  No complete rows found!")
    warmup_rows = 0

# === STEP 5: CHECK NaN PATTERNS ===
print(f"\n5. NaN PATTERN ANALYSIS")

for col in has_nans.head(5).index:
    nans = btc_features[col].isnull()
    first_valid = btc_features[col].first_valid_index()
    last_valid = btc_features[col].last_valid_index()
    
    if first_valid and last_valid:
        first_idx = btc_features.index.get_loc(first_valid)
        last_idx = btc_features.index.get_loc(last_valid)
        
        nans_start = nans.iloc[:first_idx].sum()
        nans_middle = nans.iloc[first_idx:last_idx+1].sum()
        nans_end = nans.iloc[last_idx+1:].sum()
        
        print(f"   {col}:")
        print(f"      Start: {nans_start}, Middle: {nans_middle}, End: {nans_end}")

# === STEP 6: FORWARD FILL ASSESSMENT ===
print(f"\n6. FORWARD FILL FEASIBILITY")

# Check if NaNs are only at start (good for ffill)
only_start_nans = True
for col in has_nans.index:
    first_valid = btc_features[col].first_valid_index()
    if first_valid:
        first_idx = btc_features.index.get_loc(first_valid)
        if btc_features[col].iloc[first_idx:].isnull().any():
            only_start_nans = False
            break

if only_start_nans:
    print(f"   ✓ All NaNs at start - forward fill SAFE after warmup drop")
else:
    print(f"   ⚠️  NaNs in middle of data - forward fill may hide issues")

# === STEP 7: RECOMMENDATION ===
print(f"\n{'='*60}")
print("RECOMMENDATION")
print('='*60)

if len(broken) > 0:
    print(f"1. DROP {len(broken)} broken features")

print(f"2. DROP first {warmup_rows} rows (warmup period)")
print(f"3. FORWARD FILL remaining NaNs")
print(f"4. DROP any remaining NaN rows (if any)")

expected_rows = len(btc_features) - warmup_rows
print(f"\nExpected final shape: (~{expected_rows}, {len(btc_features.columns) - len(broken)})")

# === STEP 8: EXECUTE CLEANING ===
print(f"\n{'='*60}")
print("EXECUTING CLEANING")
print('='*60)

# Drop broken features
if len(broken) > 0:
    btc_clean = btc_features.drop(columns=broken.index)
    print(f"Dropped {len(broken)} broken features")
else:
    btc_clean = btc_features.copy()

# Drop warmup
print(f"Before: {btc_clean.shape}")
btc_clean = btc_clean.iloc[warmup_rows:].copy()
print(f"After warmup drop: {btc_clean.shape}")

# Forward fill
nans_before_ff = btc_clean.isnull().sum().sum()
btc_clean = btc_clean.fillna(method='ffill')
nans_after_ff = btc_clean.isnull().sum().sum()
print(f"NaNs: {nans_before_ff:,} → {nans_after_ff:,} after ffill")

# Drop remaining NaNs
if nans_after_ff > 0:
    rows_before = len(btc_clean)
    btc_clean = btc_clean.dropna()
    print(f"Dropped {rows_before - len(btc_clean)} rows with remaining NaNs")

# === FINAL STATE ===
print(f"\n{'='*60}")
print("FINAL CLEAN DATASET")
print('='*60)
print(f"Shape: {btc_clean.shape}")
print(f"Date range: {btc_clean.index.min()} to {btc_clean.index.max()}")
print(f"Duration: {(btc_clean.index.max() - btc_clean.index.min()).days} days")
print(f"NaNs: {btc_clean.isnull().sum().sum()}")

print(f"\nSample (first 3 rows, first 10 columns):")
print(btc_clean.iloc[:3, :10])

In [4]:
# Save with memory optimization
print(f"\n{'='*60}")
print("SAVING")
print('='*60)

# Convert to float32 before saving
btc_to_save = btc_clean.copy()

for col in btc_to_save.select_dtypes(include=['float64']).columns:
    btc_to_save[col] = btc_to_save[col].astype('float32')

# Save with efficient settings
btc_to_save.to_parquet(
    'btc_futures_clean.parquet',
    engine='pyarrow',
    compression='snappy',
    index=True
)

# Free memory
del btc_to_save

print(f"✓ Saved: btc_futures_clean.parquet")
print(f"  Shape: {btc_clean.shape}")
print(f"  Memory: {btc_clean.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Verify file size
import os
file_size = os.path.getsize('btc_futures_clean.parquet') / 1e6
print(f"  File size: {file_size:.1f} MB")